In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from helpers import load_estimator_parameters, make_lagrangian_mock

from acm import setup_logging
from acm.estimators.galaxy_clustering.backends.jaxpower import (
    JaxpowerBackend,  # noqa: F401 - register backend
)
from acm.estimators.galaxy_clustering.tpcf import TwoPointCorrelationFunctionEstimator

setup_logging()

In [ ]:
los = "z"
data_positions, boxsize = make_lagrangian_mock(boxsize=500.0, los=los)
# Instanciate class
estimator = TwoPointCorrelationFunctionEstimator(
    backend='jaxpower', # We need a backend, but we won't use it
    data_positions=data_positions,
    boxsize=boxsize,
    cellsize=5.0,
)
# We don't need to set the density contrast either

# Compute the two-point correlation function with specified parameters
result = estimator.compute(
    edges = (np.arange(0, 151, 1), np.linspace(-1, 1, 120)),
    los=los,
    mode="smu",
    compute_sepsavg=False,
)

# Use the helper function to plot the result
fig, ax = estimator.plot(result)
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
# Access the data from the result object - see lsstypes docs

fig, ax = plt.subplots(figsize=(8, 6))

for ell in [0, 2, 4]:
    pole = result.project(ells=ell)
    binned_pole = pole.select(s=slice(0, None, 3))
    s = binned_pole.coords("s")
    ax.plot(s, s**2 * binned_pole, label=rf"$\ell={ell}$", alpha=0.8)

ax.set_xlabel('s [Mpc/h]')
ax.set_ylabel(r'$s^2 \xi_\ell(s)$ [Mpc/h]$^2$')
ax.set_title('Two-Point Correlation Function Multipoles')
ax.legend()
ax.grid(True, alpha=0.3)